# Pretrain experiment

Runs enhanced (unified masked) pretraining and writes everything to
`experiments/<experiment_name>/`:

- `pretrain_config.yaml` — frozen copy of the merged config (base YAML + `overrides`)
- `pretrain_overrides.yaml` — just the dict you passed below, for at-a-glance diffing
- `pretrain_metadata.json` — timestamp, git SHA, resolved device, final/best loss, wallclock, the overrides inline
- `pretrain.log` — full training log (includes GPU diagnostics at startup)
- `checkpoints/checkpoint_epoch_*.pth` — saved encoder weights
- `README.md` — name + description (the short note you write below)

Edit the **Parameters** cell, run all cells. Re-running with the same
`experiment_name` will raise unless you pass `overwrite=True`.

The `experiment_name` is the only key you need to pass to the eval notebook —
it auto-loads the architecture from this experiment's saved config.

In [ ]:
# === Parameters ===
experiment_name = "houston_enhanced_v1"
description = (
    "Houston enhanced pretraining, baseline hyperparameters from "
    "configs/pretrain/houston_pretrain_enhanced.yaml."
)

# Path to the YAML config to start from.
config_path = "configs/pretrain/houston_pretrain_enhanced.yaml"

# Dict-shaped overrides applied (deep-merge) on top of the loaded YAML.
# Multiple sections can be combined in one dict. Anything you set here is
# also recorded verbatim to experiments/<name>/pretrain_overrides.yaml so
# it's obvious what changed from the base config.
#
# Uncomment any of the blocks below to experiment.
overrides = {
    # --- Training schedule ---
    # "pretrain": {"epochs": 1500, "warmup_epochs": 80, "batch_size": 128},

    # --- Masking ratios ---
    # "pretrain": {"band_mask_ratio": 0.75, "spatial_mask_ratio": 0.25},

    # --- Loss weighting (center-weighted reconstruction) ---
    # "pretrain": {"recon_center_sigma": 2.0},

    # --- Optimizer ---
    # "pretrain": {"lr": 3e-4, "weight_decay": 0.1, "adam_betas": [0.9, 0.999],
    #              "warmup_start_factor": 0.001, "grad_clip": 0.5},

    # --- Model capacity (must match in eval; eval auto-reads these) ---
    # "model": {"embed_dim": 192, "num_heads": 4, "num_layers": 6,
    #           "proj_hidden_dim": 768, "proj_l2_normalize": False},

    # --- Hardware ---
    # "hardware": {"device": "auto", "deterministic": False},
    # "data":     {"num_workers": 8, "persistent_workers": True, "prefetch_factor": 4},

    # --- Combine multiple sections in one dict ---
    # "pretrain": {"epochs": 1500, "lr": 3e-4},
    # "model":    {"embed_dim": 192},
}

# Resume from a checkpoint inside this experiment (or another).
# Pass an absolute or repo-relative path, or None to start fresh.
resume = None

overwrite = False

In [ ]:
import os, sys
from pathlib import Path

# Make repo root importable regardless of where Jupyter was launched.
REPO = Path.cwd()
while not (REPO / "lib" / "experiments.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

In [ ]:
from lib.pretrain_runner import run_pretrain

exp = run_pretrain(
    name=experiment_name,
    description=description,
    config=config_path,
    overrides=overrides,
    resume=resume,
    overwrite=overwrite,
)
print("Experiment dir:", exp.root)

In [ ]:
# Verify what actually got recorded for this run: the resolved device
# (cuda:0 / cpu) and the overrides exactly as they were saved to disk.
import json
print("Resolved device:", exp.metadata.get("resolved_device"),
      "|", exp.metadata.get("cuda_device_name", ""))
print("Overrides applied:")
print(json.dumps(exp.metadata.get("overrides", {}), indent=2))
print("\nFull metadata:")
print(json.dumps(exp.metadata, indent=2, default=str))

In [ ]:
# List of checkpoints written by this run.
for p in sorted(exp.checkpoints_dir.glob("*.pth")):
    print(p.name, f"{p.stat().st_size/1e6:.1f} MB")